<a href="https://colab.research.google.com/github/PalakMallik/deep-learning/blob/main/next_word_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Next - word predictor  -  "quotes by famous scientists" dataset**

## 1. Load libraries and frameworks

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load dataset

In [ ]:
!wget -O qoute_dataset.csv https://raw.githubusercontent.com/AkarshVyas/Next_word_prediction/main/qoute_dataset.csv

--2026-08-17 22:26:04--  https://raw.githubusercontent.com/AkarshVyas/Next_word_prediction/main/qoute_dataset.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 535753 (523K) [text/plain]
Saving to: ‘qoute_dataset.csv’

qoute_dataset.csv   100%[===================>] 523.20K  --.-KB/s    in 0.03s   

2026-08-17 22:26:05 (17.8 MB/s) - ‘qoute_dataset.csv’ saved [535753/535753]



In [ ]:
df = pd.read_csv("qoute_dataset.csv")

df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [ ]:
print(df.shape)

(3038, 2)


In [ ]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [ ]:
# Extract "quote" column (as it is only required here)
quotes = df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


## 3. Pre-process data

### **3.1 Text cleaning**

In [ ]:
# lowercase
quotes = quotes.str.lower()

In [ ]:
# remove punctuations
import string
translator = str.maketrans("", "", string.punctuation)

quotes = quotes.apply(lambda x : x.translate(translator))

In [ ]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


### **3.2 Tokenization / Vocabulary**

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

vocab_size = 10000

tokenizer = Tokenizer(num_words = vocab_size)
tokenizer.fit_on_texts(quotes)

In [ ]:
word_index = tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

### **3.3 Vectorization**

In [ ]:
encoded_docs = tokenizer.texts_to_sequences(quotes)
print(encoded_docs[0])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]


In [ ]:
for i in range(3):
  print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [ ]:
for i in range(3):
  print(encoded_docs[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


### **3.4 Creating input-output sequences (training samples)**

In [ ]:
# sequence generation / sequence preprocessing

X = []
y = []

for seq in encoded_docs:
  for i in range(1, len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)

# At every step, the current word is the output, and all previous words are the input

In [ ]:
X

[[713],
 [713, 62],
 [713, 62, 29],
 [713, 62, 29, 19],
 [713, 62, 29, 19, 16],
 [713, 62, 29, 19, 16, 946],
 [713, 62, 29, 19, 16, 946, 10],
 [713, 62, 29, 19, 16, 946, 10, 7],
 [713, 62, 29, 19, 16, 946, 10, 7, 5],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104],
 [713,
  62,
  29,
  19,
  16,
  946,
  10,
  7,
  5,
  1156,
  8,
  70,
  293,
  10,
  145,
  12,
  809,
  104,
  752],
 [713,
  62,
  29,
  19,
  16,
  946,
  10,
  7,
  5,
  1156,
  8,
  70,
  293,
  10,
  145,
  12,
  809,
  

In [ ]:
print(len(X))
print(len(y))

85271
85271


### **3.5 Padding**

In [ ]:
max_length = max(len(x) for x in X)  # find maximum length
print(max_length)

745


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_padded = pad_sequences(X, maxlen = max_length, padding = 'pre')

In [ ]:
print(X_padded[0])   # 744 zeros and 1 number

[  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   

In [ ]:
print(X[0])

[713]


### **3.6 Convert output (y) to array**

In [ ]:
y = np.array(y)

In [ ]:
# see shapes

print(X_padded.shape)
print(y.shape)

(85271, 745)
(85271,)


### **3.7 One-hot encode the target (y)**

In [ ]:
import tensorflow as tf
y_one_hot = tf.keras.utils.to_categorical(y, num_classes=vocab_size)

In [ ]:
print(y_one_hot.shape)

(85271, 10000)


In [ ]:
print(f"Total unique words in the dataset: {len(word_index)}")

Total unique words in the dataset: 8978


## **4. Model**

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense

In [ ]:
embedding_dim = 50
rnn_units = 128

In [ ]:
rnn_model = Sequential()

rnn_model.add(         # Embedding converts words/tokens into dense numerical vectors that capture their semantic relationships (its meaning or relationships with other words)
    Embedding(input_dim = vocab_size, output_dim = embedding_dim, input_length = max_length)
)

rnn_model.add(
    SimpleRNN(units = rnn_units)
)

rnn_model.add(
    Dense(units = vocab_size, activation = "softmax")
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
rnn_model.compile(
    optimizer = "adam",
    loss = "categorical_crossentropy",
    metrics = ["accuracy"]
)

In [ ]:
lstm_model = Sequential()

lstm_model.add(
    Embedding(input_dim = vocab_size, output_dim = embedding_dim, input_length = max_length)
)

lstm_model.add(
    LSTM(units = rnn_units)
)

lstm_model.add(
    Dense(units = vocab_size, activation = "softmax")
)

In [ ]:
lstm_model.compile(
    optimizer = "adam",
    loss = "categorical_crossentropy",
    metrics = ["accuracy"]
)

In [ ]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
epochs = 10
batch_size = 128

In [ ]:
history_rnn = rnn_model.fit(
    X_padded, y_one_hot,
    epochs = epochs,
    batch_size = batch_size,
    validation_split = 0.1,
    verbose = 1
)

Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 47s 70ms/step - accuracy: 0.0301 - loss: 7.4862 - val_accuracy: 0.0393 - val_loss: 6.7632
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.0441 - loss: 6.6518 - val_accuracy: 0.0319 - val_loss: 7.7480
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.0473 - loss: 6.7821 - val_accuracy: 0.0469 - val_loss: 6.6942
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 64ms/step - accuracy: 0.0521 - loss: 6.6297 - val_accuracy: 0.0492 - val_loss: 6.6907
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.0602 - loss: 6.3263 - val_accuracy: 0.0575 - val_loss: 6.6783
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.0615 - loss: 6.4940 - val_accuracy: 0.0599 - val_loss: 6.8640
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.0692 - loss: 6.3747 - val_accuracy: 0.0633 - val_loss: 6.7254
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.0810 - loss: 6.1194 - 

In [ ]:
history_lstm = lstm_model.fit(
    X_padded, y_one_hot,
    epochs = 100,
    batch_size = batch_size,
    validation_split = 0.1,
    verbose = 1
)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 57ms/step - accuracy: 0.0391 - loss: 6.7589 - val_accuracy: 0.0426 - val_loss: 6.6920
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 37s 56ms/step - accuracy: 0.0585 - loss: 6.3239 - val_accuracy: 0.0712 - val_loss: 6.5458
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 54ms/step - accuracy: 0.0824 - loss: 6.0396 - val_accuracy: 0.0875 - val_loss: 6.4472
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.0994 - loss: 5.8229 - val_accuracy: 0.0956 - val_loss: 6.4183
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.1104 - loss: 5.6373 - val_accuracy: 0.1038 - val_loss: 6.3968
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.1202 - loss: 5.4621 - val_accuracy: 0.1064 - val_loss: 6.4120
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.1294 - loss: 5.3013 - val_accuracy: 0.1082 - val_loss: 6.4422
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.1373 - loss: 5

## **5. Save the model**

In [ ]:
lstm_model.save("lstm_model.h5")